# 02 — Step 1: Selectivity Score Screening

**Goal**: For each of the 57×24 = 1,368 attention heads, compute a Reflection Selectivity Score:

$$S(h, l) = \text{CRA}_{\text{mirror}}(h, l) - \text{CRA}_{\text{nonmirror}}(h, l)$$

This is the **screening pass** — we compute CRA scalars without storing full attention maps.

In [ ]:
# Clone repo and set up paths (run once on Colab)
!git clone https://github.com/iker0/mi-mirror.git /content/mi-mirror 2>/dev/null || echo "Already cloned"
%cd /content/mi-mirror

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import json
import pickle
from pathlib import Path
from PIL import Image

from scripts.config import (
    MIRROR_PROMPTS, NON_MIRROR_PROMPTS, SEEDS, PROMPT_OBJECTS,
    EXPERIMENTS_DIR, NUM_BLOCKS, NUM_HEADS, NUM_INFERENCE_STEPS,
    TOP_K_CANDIDATES, MIRROR_DIR, NON_MIRROR_DIR,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_with_cra
from scripts.attention_extraction import AttentionData
from scripts.metrics import compute_selectivity_matrix, rank_candidates
from scripts.visualization import plot_selectivity_heatmap, plot_top_candidates

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP1_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prompts: {len(MIRROR_PROMPTS)} mirror, {len(NON_MIRROR_PROMPTS)} non-mirror")
print(f"Seeds: {SEEDS}")
print(f"Using CLIPSeg ROIs per image")

In [ ]:
# Load model
pipe = load_flux_pipeline(quantize_nf4=False, cpu_offload=True)
print("Model loaded.")

In [ ]:
# Run CRA extraction on mirror prompts
print("=== Mirror Prompts ===")
mirror_attention_data = []

for i, prompt in enumerate(MIRROR_PROMPTS):
    obj_name = PROMPT_OBJECTS[i]
    for seed in SEEDS:
        print(f"  Prompt {i} ({obj_name}), seed {seed}...", end=" ")
        # Load pre-generated image for CLIPSeg ROI
        img_path = MIRROR_DIR / f"prompt{i:02d}_seed{seed}.png"
        image = Image.open(img_path)
        obj_roi, ref_roi = get_object_and_reflection_rois(image, obj_name)
        attn_data = AttentionData()
        image, attn_data = generate_with_cra(
            pipe, prompt, seed, obj_roi, ref_roi, attn_data
        )
        mirror_attention_data.append(attn_data)
        print(f"done ({len(attn_data.cra_scalars)} CRA values, obj={obj_roi.size}t ref={ref_roi.size}t)")

print(f"\nCollected {len(mirror_attention_data)} mirror samples")

In [ ]:
# Run CRA extraction on non-mirror prompts
print("=== Non-Mirror Prompts ===")
nonmirror_attention_data = []

for i, prompt in enumerate(NON_MIRROR_PROMPTS):
    obj_name = PROMPT_OBJECTS[i]
    for seed in SEEDS:
        print(f"  Prompt {i} ({obj_name}), seed {seed}...", end=" ")
        # Load pre-generated image for CLIPSeg ROI
        img_path = NON_MIRROR_DIR / f"prompt{i:02d}_seed{seed}.png"
        image = Image.open(img_path)
        obj_roi, ref_roi = get_object_and_reflection_rois(image, obj_name)
        attn_data = AttentionData()
        image, attn_data = generate_with_cra(
            pipe, prompt, seed, obj_roi, ref_roi, attn_data
        )
        nonmirror_attention_data.append(attn_data)
        print(f"done ({len(attn_data.cra_scalars)} CRA values, obj={obj_roi.size}t ref={ref_roi.size}t)")

print(f"\nCollected {len(nonmirror_attention_data)} non-mirror samples")

In [ ]:
# Compute selectivity matrix
selectivity = compute_selectivity_matrix(mirror_attention_data, nonmirror_attention_data)
print(f"Selectivity matrix shape: {selectivity.shape}")
print(f"Max selectivity: {selectivity.max():.6f}")
print(f"Min selectivity: {selectivity.min():.6f}")
print(f"Mean selectivity: {selectivity.mean():.6f}")

In [ ]:
# Visualize selectivity heatmap
import matplotlib.pyplot as plt

fig = plot_selectivity_heatmap(selectivity)
fig.savefig(STEP1_DIR / "selectivity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank and display top candidates
candidates = rank_candidates(selectivity, top_k=TOP_K_CANDIDATES)
print(f"\nTop-{TOP_K_CANDIDATES} Candidate Reflection Heads:")
print(f"{'Rank':<6} {'Block':<8} {'Head':<8} {'Selectivity':<12}")
print("-" * 34)
for rank, (b, h, s) in enumerate(candidates, 1):
    stream = "dual" if b < 19 else "single"
    print(f"{rank:<6} {b:<8} {h:<8} {s:<12.6f}  ({stream})")

In [ ]:

fig = plot_top_candidates(candidates)
fig.savefig(STEP1_DIR / "top_candidates.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save results
results = {
    "selectivity_matrix": selectivity,
    "candidates": candidates,
    "mirror_attention_data": mirror_attention_data,
    "nonmirror_attention_data": nonmirror_attention_data,
}
with open(STEP1_DIR / "step1_results.pkl", "wb") as f:
    pickle.dump(results, f)

# Also save candidates as JSON for easy reference
candidates_json = [
    {"block": b, "head": h, "selectivity": s}
    for b, h, s in candidates
]
with open(STEP1_DIR / "candidates.json", "w") as f:
    json.dump(candidates_json, f, indent=2)

print(f"Results saved to {STEP1_DIR}")